DEFINE AND VISUALISE REGIONS OF INTEREST

In [155]:
import pyvista
import pyvistaqt
import nibabel
import mne
from mne.viz import Brain
import os
import numpy as np
import pickle

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config import check_paths

In [190]:
HEMI = "-lh"  # or "rh" for right hemisphere
ROI = {
    "M1": [
        f"G_precentral{HEMI}",
        # f"S_central{HEMI}", # divides the precentral (motor) and postcentral (sensory) gyrus
    ],
    "S1": [
        f"G_postcentral{HEMI}",
        f"S_postcentral{HEMI}",
        ],
    "PMC": [
        f"S_precentral-sup-part{HEMI}",
        f"S_precentral-inf-part{HEMI}",
        # f"G_front_middle{HEMI}",
        f"G_front_inf-Opercular{HEMI}",
        ],
    "SMA": [
        f"G_and_S_paracentral{HEMI}",
        f"G_front_sup{HEMI}", # extends a bit too far front: only posterior parts belong to SMA and PMC
        ],
}

colors = {
    "M1": "red",
    "S1": "blue",
    "PMC": "green",
    "SMA": "purple",
}

In [ ]:
mris_dir = "D:\\BonoKat\\research project\\# PD project\\fs"
fs_sub = "fsaverage_bem"   # or your participant

labels = mne.read_labels_from_annot(
    subject=fs_sub,
    parc="aparc.a2009s", # "aparc.a2009s", "Yeo2011_7Networks_N1000", "Yeo2011_17Networks_N1000"
    hemi=HEMI[1:],
    subjects_dir=mris_dir
)

label_dict = {label.name: label for label in labels}

ROI_labels = {
    roi: {
        label_name: label_dict[label_name]
        for label_name in label_names
    }
    for roi, label_names in ROI.items()
}

for label in labels:
    print(label.name)

# # OPTIONAL: plot label borders on the brain surface
# brain = Brain(
#     subject,
#     hemi=HEMI[1:],
#     surf="inflated",
#     subjects_dir=mris_dir,
#     background="black"
# )

# for label in labels:
#     print(label)
#     brain.add_label(label, borders=True)


In [89]:
brain = Brain(
    fs_sub,
    hemi=HEMI[1:],
    surf="pial",
    subjects_dir=mris_dir,
    background="black"
)

for roi, label_names in ROI.items():
    for label_name in label_names:
        brain.add_label(label_dict[label_name], borders=False, color=colors[roi])


For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
G_precentral-lh
G_postcentral-lh
S_postcentral-lh
S_precentral-sup-part-lh
S_precentral-inf-part-lh
G_front_inf-Opercular-lh
G_and_S_paracentral-lh
G_front_sup-lh


EXTRACT SOURCE TIME SERIES BY LABEL

In [99]:
FS_FOLDER = "D:\\BonoKat\\research project\\# PD project\\fs"
FS_SUB = "fsaverage_bem"
FS_SRC_PATH = os.path.join(FS_FOLDER, FS_SUB, "bem", f"{FS_SUB}-ico4-src.fif")
src = mne.read_source_spaces(FS_SRC_PATH)


    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    [done]
    2 source spaces read


In [91]:
source_path = "F:\\# study 2\\eeg_data\\epochs\\source"
group = "Y"
sub = "s1_pac_sub24"
task = "FTT"
stage = "plan"
source_dir = "F:\\# study 2\\eeg_data\\epochs\\source"
source_path = os.path.join(source_dir, group, sub, task, stage)

epo_stcs = [mne.read_source_estimate(os.path.join(source_path, fname_stc)) for fname_stc in os.listdir(source_path) if fname_stc.endswith('-lh.stc')]
epo_stcs[0]

<SourceEstimate | 5124 vertices, tmin : -500.0 (ms), tmax : 500.0 (ms), tstep : 2.0 (ms), data shape : (5124, 501), ~9.8 MiB>

In [134]:
ROI_labels

{'M1': {'G_precentral-lh': <Label | fsaverage_bem, 'G_precentral-lh', lh : 3876 vertices>},
 'S1': {'G_postcentral-lh': <Label | fsaverage_bem, 'G_postcentral-lh', lh : 3509 vertices>,
  'S_postcentral-lh': <Label | fsaverage_bem, 'S_postcentral-lh', lh : 4447 vertices>},
 'PMC': {'S_precentral-sup-part-lh': <Label | fsaverage_bem, 'S_precentral-sup-part-lh', lh : 1846 vertices>,
  'S_precentral-inf-part-lh': <Label | fsaverage_bem, 'S_precentral-inf-part-lh', lh : 1881 vertices>,
  'G_front_inf-Opercular-lh': <Label | fsaverage_bem, 'G_front_inf-Opercular-lh', lh : 1800 vertices>},
 'SMA': {'G_and_S_paracentral-lh': <Label | fsaverage_bem, 'G_and_S_paracentral-lh', lh : 2272 vertices>,
  'G_front_sup-lh': <Label | fsaverage_bem, 'G_front_sup-lh', lh : 8394 vertices>}}

In [192]:
for roi, labels in ROI_labels.items():

    print(f"Processing ROI: {roi}")

    for label_name, label in labels.items():
        print(f"  Processing label: {label_name}")

Processing ROI: M1
  Processing label: G_precentral-lh
Processing ROI: S1
  Processing label: G_postcentral-lh
  Processing label: S_postcentral-lh
Processing ROI: PMC
  Processing label: S_precentral-sup-part-lh
  Processing label: S_precentral-inf-part-lh
  Processing label: G_front_inf-Opercular-lh
Processing ROI: SMA
  Processing label: G_and_S_paracentral-lh
  Processing label: G_front_sup-lh


In [203]:
roi_data = {}

for roi, labels in ROI_labels.items():

    roi_data[roi] = {}

    for label_name, label in labels.items():

        stcs_label = [stc.in_label(label) for stc in epo_stcs]

        roi_data[roi][label_name] = {
            "data": np.stack([stc.data for stc in stcs_label]),
            "vertices": stcs_label[0].vertices[0],
            "hemisphere": "lh",
            "times": stcs_label[0].times,
            "tstep": stcs_label[0].tstep,
            "sfreq": stcs_label[0].sfreq,
        }

roi_data

{'M1': {'G_precentral-lh': {'data': array([[[-5.89796007e-02, -3.00747007e-02,  3.81895974e-02, ...,
             3.06706190e-01,  1.86369091e-01,  9.67492461e-02],
           [-1.48766249e-01, -2.66776472e-01, -2.27447122e-01, ...,
            -1.67692900e-01,  2.05570832e-01,  4.35404718e-01],
           [-5.16671538e-02, -4.08505760e-02, -4.91171740e-02, ...,
            -1.46756083e-01, -1.47479475e-01, -1.12603329e-01],
           ...,
           [-4.74931411e-02, -3.80117167e-03,  3.27860229e-02, ...,
             2.73040563e-01,  2.07324013e-01,  1.65326655e-01],
           [ 3.17656666e-01,  2.86941260e-01,  1.35788128e-01, ...,
            -5.77007011e-02, -2.94542294e-02, -5.50838895e-02],
           [ 8.05290937e-02,  7.66360164e-02,  1.87161155e-02, ...,
            -4.71026786e-02,  5.70299067e-02,  1.84942141e-01]],
   
          [[ 1.83201700e-01,  1.68715015e-01,  1.19594626e-01, ...,
            -2.65904903e-01, -3.12137097e-01, -3.19755942e-01],
           [-2.3669916

In [204]:
roi_dir= os.path.join("F:\\# study 2\\eeg_data\\epochs\\roi_stcs", group, task, stage)
check_paths(roi_dir)

with open(os.path.join(roi_dir, f"{sub}_{task}_{stage}_roi_data.pkl"), "wb") as f:
    pickle.dump(roi_data, f)


VIZ

In [189]:
roi_data

{'M1': {'G_precentral-lh': {'data': array([[[-5.89796007e-02, -3.00747007e-02,  3.81895974e-02, ...,
             3.06706190e-01,  1.86369091e-01,  9.67492461e-02],
           [-1.48766249e-01, -2.66776472e-01, -2.27447122e-01, ...,
            -1.67692900e-01,  2.05570832e-01,  4.35404718e-01],
           [-5.16671538e-02, -4.08505760e-02, -4.91171740e-02, ...,
            -1.46756083e-01, -1.47479475e-01, -1.12603329e-01],
           ...,
           [-4.74931411e-02, -3.80117167e-03,  3.27860229e-02, ...,
             2.73040563e-01,  2.07324013e-01,  1.65326655e-01],
           [ 3.17656666e-01,  2.86941260e-01,  1.35788128e-01, ...,
            -5.77007011e-02, -2.94542294e-02, -5.50838895e-02],
           [ 8.05290937e-02,  7.66360164e-02,  1.87161155e-02, ...,
            -4.71026786e-02,  5.70299067e-02,  1.84942141e-01]],
   
          [[ 1.83201700e-01,  1.68715015e-01,  1.19594626e-01, ...,
            -2.65904903e-01, -3.12137097e-01, -3.19755942e-01],
           [-2.3669916

In [195]:
import matplotlib.pyplot as plt
%matplotlib qt

for roi, labels in roi_data.items():

    for label_name, label_data in labels.items():

        data = label_data["data"]      # epochs x vertices x times
        times = label_data["times"]

        # average across epochs and vertices
        tc = data.mean(axis=(0, 1))

        fig, ax = plt.subplots(figsize=(8, 3))

        ax.plot(times, tc)
        ax.axvline(0, color="k", linestyle="--")
        ax.set_ylim(-0.03, 0.05)

        ax.set_title(f"{roi}: {label_name}")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Source amplitude")

        plt.show()

ERPAC

In [197]:
import tensorpac
import mne
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorpac import EventRelatedPac
from tensorpac.signals import pac_signals_wavelet
import scipy
from mne.stats import combine_adjacency
from mne.stats import fdr_correction

%matplotlib qt

In [ ]:
# file check
with open(os.path.join(roi_dir, f"{sub}_{task}_{stage}_roi_data.pkl"), "rb") as f:
    roi_data = pickle.load(f)

roi_data

{'M1': {'G_precentral-lh': {'data': array([[[-5.89796007e-02, -3.00747007e-02,  3.81895974e-02, ...,
             3.06706190e-01,  1.86369091e-01,  9.67492461e-02],
           [-1.48766249e-01, -2.66776472e-01, -2.27447122e-01, ...,
            -1.67692900e-01,  2.05570832e-01,  4.35404718e-01],
           [-5.16671538e-02, -4.08505760e-02, -4.91171740e-02, ...,
            -1.46756083e-01, -1.47479475e-01, -1.12603329e-01],
           ...,
           [-4.74931411e-02, -3.80117167e-03,  3.27860229e-02, ...,
             2.73040563e-01,  2.07324013e-01,  1.65326655e-01],
           [ 3.17656666e-01,  2.86941260e-01,  1.35788128e-01, ...,
            -5.77007011e-02, -2.94542294e-02, -5.50838895e-02],
           [ 8.05290937e-02,  7.66360164e-02,  1.87161155e-02, ...,
            -4.71026786e-02,  5.70299067e-02,  1.84942141e-01]],
   
          [[ 1.83201700e-01,  1.68715015e-01,  1.19594626e-01, ...,
            -2.65904903e-01, -3.12137097e-01, -3.19755942e-01],
           [-2.3669916

In [230]:
erpac_results = {
    "theta_gamma": {},
    "alpha_gamma": {},
    "beta_gamma": {},
}

gamma = (30, 80, 5, 1)

couplings = {
    "theta_gamma": ([4, 8], gamma),
    "alpha_gamma": ([8, 12], gamma),
    "beta_gamma": ([13, 30], gamma),
}

sf = roi_data["M1"]["G_precentral-lh"]["sfreq"] # extract from any ROI and label, since they all have the same sampling frequency
alpha_threshold = 0.05


In [ ]:
erpac_results = {}

for coupling_name, (phase_freq, amp_freq) in couplings.items():

    print(f"\nComputing {coupling_name}")

    p = EventRelatedPac(
        f_pha=phase_freq,
        f_amp=amp_freq
    )

    erpac_results[coupling_name] = {}

    for roi, labels in roi_data.items():

        erpac_results[coupling_name][roi] = {}

        for label_name, label_data in labels.items():

            data = label_data["data"]      # (epochs, vertices, times)

            n_epochs, n_vertices, n_times = data.shape
            n_freqs = len(p.yvec)

            vertex_erpac = np.zeros((n_vertices, n_freqs, n_times))
            vertex_sig = np.zeros((n_vertices, n_times))

            for v in range(n_vertices):

                x = data[:, v, :]

                erpac = p.filterfit(
                    sf,
                    x,
                    method="circular",
                    mcp="bonferroni"
                ).squeeze()

                pvalues = p.pvalues.squeeze()

                sig_mask = pvalues <= alpha_threshold

                sig_mean = np.where(
                    np.any(sig_mask, axis=0),
                    np.nanmean(
                        np.where(sig_mask, erpac, np.nan),
                        axis=0
                    ),
                    0
                )

                vertex_erpac[v] = erpac
                vertex_sig[v] = sig_mean

            erpac_results[coupling_name][roi][label_name] = {
                "erpac": vertex_erpac,
                "sig_mean": vertex_sig,
                "times": label_data["times"],
                "vertices": label_data["vertices"],
            }

In [234]:
with open(os.path.join(roi_dir, f"{sub}_{task}_{stage}_erpac_results.pkl"), "wb") as f:
    pickle.dump(erpac_results, f)

In [235]:
with open(os.path.join(roi_dir, f"{sub}_{task}_{stage}_erpac_results.pkl"), "rb") as f:
    erpac_results = pickle.load(f)

In [224]:
erpac.shape

(45, 501)

In [211]:
erpac_results

{'theta_gamma': {'M1': {'G_precentral-lh': {'erpac': array([[[0.06938324, 0.04948605, 0.018336  , ..., 0.21948362,
             0.21679193, 0.20943297]],
    
           [[0.12883354, 0.1538669 , 0.03971307, ..., 0.14862419,
             0.14912871, 0.19088698]],
    
           [[0.05402646, 0.13998701, 0.11030427, ..., 0.15393339,
             0.21643539, 0.19865052]],
    
           ...,
    
           [[0.11794808, 0.0584917 , 0.0359368 , ..., 0.22865074,
             0.24625333, 0.19689674]],
    
           [[0.05084352, 0.03644759, 0.01736437, ..., 0.0742453 ,
             0.06063093, 0.06655044]],
    
           [[0.06872989, 0.04072254, 0.04150528, ..., 0.16101757,
             0.15770046, 0.13659426]]], shape=(60, 1, 501)),
    'sig_mean': array([[0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           ...,
           [0., 0., 0., ..., 0., 0., 0.],
           [0., 0., 0., ..., 0., 0., 0.],
           [0.,

In [215]:
vertex_erpac.shape

(134, 1, 501)

In [226]:
import numpy as np

roi_mean = {}

for roi, labels in erpac_results["theta_gamma"].items():

    label_means = []

    for label_name, res in labels.items():
        # average across vertices
        label_means.append(
            res["erpac"].mean(axis=0)     # (freqs, times)
        )

    # average anatomical labels
    roi_mean[roi] = np.mean(label_means, axis=0)

roi_mean["M1"].shape

(45, 501)

In [227]:
import matplotlib.pyplot as plt

roi = "M1"

plt.figure(figsize=(8,4))

plt.imshow(
    roi_mean[roi],
    aspect="auto",
    origin="lower",
    extent=[
        times[0],
        times[-1],
        gamma[0],
        gamma[1]
    ]
)

plt.colorbar(label="ERPAC")
plt.xlabel("Time (s)")
plt.ylabel("Gamma frequency (Hz)")
plt.title(f"{roi}: theta-gamma ERPAC")
plt.show()

**STOPPED HERE**
- Everything works
- trace that the scripts make sense!

In [233]:
import matplotlib.pyplot as plt
import numpy as np

for coupling in erpac_results:

    # Average ERPAC across vertices, then across anatomical labels
    roi_mean = {}

    for roi, labels in erpac_results[coupling].items():

        label_means = [
            res["erpac"].mean(axis=0)      # (gamma_freqs, times)
            for res in labels.values()
        ]

        roi_mean[roi] = np.mean(label_means, axis=0)

    # use times from any ROI
    first_roi = next(iter(erpac_results[coupling]))
    first_label = next(iter(erpac_results[coupling][first_roi]))
    times = erpac_results[coupling][first_roi][first_label]["times"]

    # Plot
    fig, axes = plt.subplots(
        2, 2,
        figsize=(10, 8),
        sharex=True,
        sharey=True
    )

    for ax, roi in zip(axes.ravel(), roi_mean):

        im = ax.imshow(
            roi_mean[roi],
            aspect="auto",
            origin="lower",
            extent=[times[0], times[-1], gamma[0], gamma[1]]
        )

        ax.set_title(roi)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Gamma (Hz)")

    fig.suptitle(coupling.replace("_", " ").title())

    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="ERPAC")

    plt.tight_layout()
    plt.show()

C:\Users\a1902989\AppData\Local\Temp\ipykernel_12984\2749291090.py:48: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
